# GörEndir — YouTube Video & Subtitle Downloader (Google Colab)

This notebook is for running **GörEndir** on **Google Colab** with downloads saved to Google Drive.

## Features
- Download single videos or entire playlists
- Multi-language subtitle extraction (SRT + TXT)
- `Dict[str, int]` input format: `{"url": start_number}`
- `reverse_download=True`: reverse playlist order (last video → #1)
- `skip_download=True`: download only subtitles, skip video file
- `playlist_end=N`: limit number of videos
- Save directly to Google Drive

## ⚠️ Colab-specific notes
1. **No SSL fix needed** — Colab connects to YouTube directly without corporate proxies.
2. **Deno required** for video downloads (`skip_download=False`). Installed in Step 2.
3. **Cookies required** — YouTube blocks unauthenticated Colab traffic. See Step 4.
4. **Google Drive** mounted in Step 5 for persistent storage.

Without Deno + cookies you will get errors like:
- `No supported JavaScript runtime could be found`
- `n challenge solving failed`
- `Sign in to confirm you're not a bot`

## Step 1: Install GörEndir + yt-dlp

In [ ]:
# Install GörEndir from GitHub + upgrade yt-dlp
!pip install -q --upgrade yt-dlp
!pip install -q --upgrade --force-reinstall git+https://github.com/MammadTavakoli/gorendir.git
!yt-dlp --version

## Step 2: Install Deno (JavaScript Runtime)

yt-dlp needs Deno to solve YouTube's JavaScript `n`-challenge.
Without it, you get: `No supported JavaScript runtime could be found` and `n challenge solving failed`.

In [ ]:
# Install Deno on Colab (Linux)
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.deno/bin") + ":" + os.environ["PATH"]
!deno --version

## Step 3: Mount Google Drive

Downloads will be saved to `/content/drive/MyDrive/YouTubeDownloads` so they persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save location on Google Drive
save_path = '/content/drive/MyDrive/YouTubeDownloads'
import os; os.makedirs(save_path, exist_ok=True)
print(f'✅ Save path: {save_path}')

## Step 4: Upload YouTube Cookies

YouTube requires authentication to avoid bot detection.
Without cookies, you will get: **"Sign in to confirm you're not a bot"**

### How to get cookies.txt:
1. Install a browser extension to export cookies:
   - **Chrome**: [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc)
   - **Firefox**: [cookies.txt](https://addons.mozilla.org/en-US/firefox/addon/cookies-txt/)
2. Log into YouTube in your browser
3. Click the extension and export cookies for youtube.com → save as `cookies.txt`
4. Upload the file using the cell below (Option A), OR place it in your Google Drive (Option B)

In [ ]:
# Option A: Upload cookies.txt directly to Colab
from google.colab import files
uploaded = files.upload()  # pick cookies.txt
cookies_path = 'cookies.txt'
import os
if os.path.exists(cookies_path):
    print(f'✅ Cookies uploaded: {cookies_path}')
else:
    print('⚠️ No cookies.txt uploaded.')

In [ ]:
# Option B (alternative): Use cookies from Google Drive
# Uncomment and set the path to cookies.txt inside your Drive.
# cookies_path = '/content/drive/MyDrive/cookies.txt'
# import os
# print(f'Using cookies from: {cookies_path} ({"found" if os.path.exists(cookies_path) else "NOT found"})')

## Step 5: Import GörEndir

In [ ]:
from gorendir import YouTubeDownloader
print('✅ gorendir imported successfully')

## Step 6: Helper Function

The `download_videos` function wraps `YouTubeDownloader.download_video()`.

### Input Format for `video_urls`

| Format | Example | Description |
|--------|---------|-------------|
| `str` | `"https://youtube.com/watch?v=XXX"` | Single URL |
| `Dict[str, int]` | `{"https://youtube.com/playlist?list=XXX": 8}` | URL with start number |
| `List[Union[str, Dict[str, int]]]` | `["url1", {"url2": 5}]` | Mixed list |

When using `Dict[str, int]`, the integer value means:
- **Skip** the first N-1 videos in download order
- **Number** files starting from that number

In [ ]:
def download_videos(video_urls, save_path, reverse_download=False, skip_download=False,
                    playlist_end=0, cookies_path=None):
    """
    Download videos using GörEndir.

    Args:
        video_urls: URL(s) to download. Can be:
            - str: Single URL
            - Dict[str, int]: {"url": start_number}  e.g. {"https://...playlist?list=XXX": 8}
            - List[Union[str, Dict[str, int]]]: Mixed list of URLs and URL-start_number dicts
        save_path: Directory to save downloads
        reverse_download: If True, reverse playlist order (last video -> #1)
        skip_download: If True, only download subtitles (skip video file)
        playlist_end: If > 0, stop after downloading this many videos
        cookies_path: Path to cookies.txt for YouTube authentication
    """
    downloader = YouTubeDownloader(
        save_directory=save_path,
        max_resolution=1080,
        subtitle_languages=["en", "fa"],
        cookies_path=cookies_path,
    )

    downloader.download_video(
        video_urls=video_urls,
        reverse_download=reverse_download,
        skip_download=skip_download,
        force_download=False,
        yt_dlp_write_subs=True,
        download_subtitles=True,
        playlist_end=playlist_end,
    )

## Step 7: Download Configurations

Define different download scenarios. Each config has:
- `name`: A label for display
- `reverse_download`: Whether to reverse playlist order
- `skip_download`: Whether to skip video (only download subtitles)
- `playlist_end`: Limit number of videos (0 = no limit)
- `urls`: The URL(s) - can be a **list of strings**, a **dict of url→start_number**, or a **mixed list**

In [ ]:
# -- Subtitle-only (reverse order) --
sbtitle_reverse_urls = {
    'name': 'sbtitle_reverse_urls',
    'reverse_download': True,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 8},
    ]
}

In [ ]:
# -- Video + Subtitle (reverse order) --
video_reverse_urls = {
    'name': 'video_reverse_urls',
    'reverse_download': True,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
    ]
}

In [ ]:
# -- Subtitle-only (normal order) --
sbtitle_urls = {
    'name': 'sbtitle_urls',
    'reverse_download': False,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 3},
    ]
}

In [ ]:
# -- Video + Subtitle (normal order) --
video_urls_config = {
    'name': 'video_urls',
    'reverse_download': False,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://www.youtube.com/watch?v=_S_af9C9z2Q",
    ]
}

## Step 8: Run Downloads

In [ ]:
video_list = [sbtitle_reverse_urls, video_reverse_urls, sbtitle_urls, video_urls_config]

for config in video_list:
    print('*' * 20, config['name'], '*' * 20)
    urls = config['urls']
    if not urls:
        print(f"  No URLs configured for {config['name']}, skipping.")
        continue

    download_videos(
        video_urls=urls,
        save_path=save_path,
        reverse_download=config['reverse_download'],
        skip_download=config['skip_download'],
        playlist_end=config.get('playlist_end', 0),
        cookies_path=cookies_path,
    )

---

## Advanced Usage Examples

In [ ]:
# Example 1: Single video
# download_videos("https://youtube.com/watch?v=dQw4w9WgXcQ", save_path,
#                 cookies_path=cookies_path)

In [ ]:
# Example 2: Dict format — skip first 7, start numbering from 8
# download_videos({"https://youtube.com/playlist?list=PLxxxxxxx": 8}, save_path,
#                 cookies_path=cookies_path)

In [ ]:
# Example 3: Mixed list
# download_videos([
#     "https://youtube.com/watch?v=abc123",
#     {"https://youtube.com/playlist?list=PLxxxx": 5},
# ], save_path, cookies_path=cookies_path)

In [ ]:
# Example 4: Reverse + start number
# download_videos(
#     {"https://youtube.com/playlist?list=PLxxxx": 8},
#     save_path,
#     reverse_download=True,
#     cookies_path=cookies_path
# )

In [ ]:
# Example 5: Limit to 10 videos
# download_videos(
#     "https://youtube.com/playlist?list=PLxxxx",
#     save_path,
#     playlist_end=10,
#     cookies_path=cookies_path
# )

In [ ]:
# Example 6: YouTubeDownloader directly with all options
# downloader = YouTubeDownloader(
#     save_directory=save_path,
#     max_resolution=720,
#     subtitle_languages=["en", "fa", "tr"],
#     retry_attempts=5,
#     cookies_path=cookies_path,
# )
# downloader.download_video(
#     video_urls={"https://youtube.com/playlist?list=PLxxxx": 3},
#     reverse_download=False,
#     skip_download=False,
#     force_download=True,
#     yt_dlp_write_subs=True,
#     download_subtitles=True,
#     playlist_end=0,
# )